## Action Responder Prompt

This notebook hopes to:

- Gather trace data for action-responder agent
- Structure and save as mlflow dataset
- Utilize judges to evaluate current agent
- Try with better model to confirm judges are working
- Run GEPA optimzation to improve

In [1]:
import mlflow
from mlflow.tracking import MlflowClient

from summit_sim.settings import settings

mlflow.tracing.enable_notebook_display()

client = MlflowClient()
mlflow.set_tracking_uri(settings.mlflow_tracking_uri)
mlflow.set_experiment(settings.mlflow_experiment_name)
experiment = client.get_experiment_by_name(settings.mlflow_experiment_name)

In [2]:
# Get all traces from last 7 days (adjust as needed)
filter_string = """
tags.graph_type = 'simulation' AND
tags.agent_name = 'action-responder'
"""
traces = client.search_traces(
    locations=[experiment.experiment_id],
    filter_string=filter_string,
    max_results=500,
)

2026-03-30 08:44:29 [WARNING] urllib3.connectionpool: Connection pool is full, discarding connection: mlflow.bhamm-lab.com. Connection pool size: 10
2026-03-30 08:44:29 [WARNING] urllib3.connectionpool: Connection pool is full, discarding connection: mlflow.bhamm-lab.com. Connection pool size: 10
2026-03-30 08:44:29 [WARNING] urllib3.connectionpool: Connection pool is full, discarding connection: mlflow.bhamm-lab.com. Connection pool size: 10
2026-03-30 08:44:29 [WARNING] urllib3.connectionpool: Connection pool is full, discarding connection: mlflow.bhamm-lab.com. Connection pool size: 10
2026-03-30 08:44:29 [WARNING] urllib3.connectionpool: Connection pool is full, discarding connection: mlflow.bhamm-lab.com. Connection pool size: 10
2026-03-30 08:44:29 [WARNING] urllib3.connectionpool: Connection pool is full, discarding connection: mlflow.bhamm-lab.com. Connection pool size: 10
2026-03-30 08:44:29 [WARNING] urllib3.connectionpool: Connection pool is full, discarding connection: mlfl

In [3]:
spans = [
    span
    for trace in traces
    for span in client.get_trace(trace.info.trace_id).data.spans
    if span.name == "action_response_agent"
]

records = [
    {"inputs": span.inputs["input_data"], "outputs": span.outputs} for span in spans
]

dataset = client.create_dataset(
    name="action_response_agent",
    experiment_id=[experiment.experiment_id],
)
dataset.merge_records(records)

<EvaluationDataset: experiment_ids=['7'], profile=None, records=[DatasetRecord(dataset_id='d-1c216a47f8de4dd7b8171702966fea78',
               inputs={'hidden_state': 'Patient is A&O x2. HR 102, RR 20, BP '
                                       '128/76. SCTM: Pale, cool, moist. '
                                       'Obvious swelling and deformity over '
                                       'right clavicle. Temperatures of the '
                                       'head and body are normal. Head injury '
                                       'with brief disorientation, no loss of '
                                       'consciousness. No airway compromise, '
                                       'no bleeding from mouth or nose. '
                                       'SAMPLE: Signs of trauma, no known '
                                       'allergies, no medications, no '
                                       'significant past medical history, last '
                    

[Trace(trace_id=tr-c71898cb8a264b9dc72985abfa377e86), Trace(trace_id=tr-2991fa99049794142e94d49f4af906d4), Trace(trace_id=tr-caddbe73aafe4d0d18618ece705ada7d), Trace(trace_id=tr-f4ba74c0195730b61cc4ad2f45439124), Trace(trace_id=tr-d003be5037650bcd7df7b900f3e44054), Trace(trace_id=tr-70ec6a42efc3fa75fc4bff1c14fcadb9), Trace(trace_id=tr-f7f716b149bedc367a9f6e8c2edf78d4), Trace(trace_id=tr-63adcf98c6d9413dcfe6eb54386af58f), Trace(trace_id=tr-a0626c4ee96e5ce719ee39561dff654f), Trace(trace_id=tr-4f9541428cffa543b375e4ee834da4f4)]

In [4]:
from mlflow.genai.judges import make_judge

JUDGE_MODEL_ENDPOINT = "openrouter:/deepseek/deepseek-v3.2"

COMPLETION_JUDGE_INSTRUCTIONS = """\
You are evaluating the scoring accuracy and pedagogical quality of an
AI-generated response in a wilderness first responder (WFR) training simulation.

=== PATIENT ASSESSMENT SYSTEM (PAS) - GUIDELINES ===

The PAS follows this general order, but students may bundle steps efficiently or adapt based on the situation:

1. SCENE SIZE-UP: 0-.2 points
   - Check for hazards (environmental dangers, unstable terrain, etc.)
   - Identify mechanism of injury (MOI)
   - Count patients and assess available resources

2. PRIMARY ASSESSMENT (ABCDE): 0-.2 points
   - A: Airway assessment
   - B: Breathing evaluation
   - C: Circulation (pulse, bleeding, shock signs)
   - D: Disability (mental status, AVPU)
   - E: Exposure/Environment (clothing, temperature, elements)

3. SECONDARY ASSESSMENT: 0-.2 points
   - Vital signs (HR, RR, BP, SCTM, temperature)
   - Head-to-Toe exam (systematic physical check)
   - SAMPLE history (Signs/Symptoms, Allergies, Medications, Past history, Last intake, Events leading to injury)

4. TREATMENT: 0-.2 points
   - Address immediate life threats
   - Immobilize injuries
   - Wound care
   - Pain management

5. EVACUATION PLAN: 0-.2 points
   - Stay vs. Go decision
   - Resource planning
   - Timeline establishment

AI Input:
{{ inputs }}

DO NOT CONSIDER PREVIOUS INPUTS
Only evaluate the final AI agent output below:
{{ outputs }}


Evaluate against these criteria:
1. Does completion_score align with the WFR rubric based on cumulative actions?
2. Is the score increase from previous turn reasonable (<=0.2 unless explicit bundling)?
3. Does the score always build on itself? Does completion_score never decrease across turns?
4. Is the agent being overtly strict and not giving an increased score when a new action is taken?
5. If there was no action from the student, the completion_score should not increase from it's previous turn.

Provide a score of 0-1 on how well the AI Agent performs. 1 being the agent did perfect; 0 implying the agent got everything wrong.
"""


completion_judge = make_judge(
    name="completion-judge",
    instructions=COMPLETION_JUDGE_INSTRUCTIONS,
    model=JUDGE_MODEL_ENDPOINT,
    feedback_value_type=float,
)

In [5]:
QUALITY_JUDGE_INSTRUCTIONS = """\
You are evaluating the structure and quality of an AI-generated response \
in a wilderness first responder training simulation.

AI Input:
{{ inputs }}

DO NOT CONSIDER PREVIOUS INPUTS
Only evaluate the final AI agent output below:
{{ outputs }}


Evaluate against these criteria:
1. Does narrative_text end with an open question? Does feedback contain NO questions?
2. Is the feedback personalized to the students response and focus only on history, not future?
3. Is the feedback encouraging and constructive without harsh corrections and focused
    on feedback alone?
4. Is feedback and narrative length between 2-4 sentences?
5. If the student indicated some action like 'check vitals', was some information revealed like 'heart rate 120'?
6. Does the narrative not overtly share hidden information, only revealing if the student's action constitutes it?

Provide a score of 0-1 on how well the AI Agent performs. 1 being the agent did perfect; 0 implying the agent got everything wrong.
"""


quality_judge = make_judge(
    name="quality-judge",
    instructions=QUALITY_JUDGE_INSTRUCTIONS,
    model=JUDGE_MODEL_ENDPOINT,
    feedback_value_type=float,
)

In [6]:
MEDICAL_JUDGE_INSTRUCTIONS = """\
You are evaluating the medical accuracy of an AI-generated response in a \
wilderness first responder training simulation.

AI Input:
{{ inputs }}

DO NOT CONSIDER PREVIOUS INPUTS
Only evaluate the final AI agent output below:
{{ outputs }}


TIP: Reference hidden_truth and hidden_state to determine if treatment was premature.

Evaluate against these criteria:
1. Is was_correct accurate?
 - was_correct should be FALSE if student performed treatment \
    (splint, bandage, medication, move patient) without assessment
 - was_correct should be TRUE for assessment actions (vitals, exam, SAMPLE history)
2. Does was_correct accurately gage the quality of the student's action?
3. Is there anything in the AI response that is not medically accurate?

Provide a score of 0-1 on how well the AI Agent performs. 1 being the agent did perfect; 0 implying the agent got everything wrong.
"""


medical_judge = make_judge(
    name="medical-judge",
    instructions=MEDICAL_JUDGE_INSTRUCTIONS,
    model=JUDGE_MODEL_ENDPOINT,
    feedback_value_type=float,
)

In [10]:
scorers = [completion_judge, quality_judge, medical_judge]
# dataset = client.get_dataset(dataset_id="d-1c216a47f8de4dd7b8171702966fea78")

results = mlflow.genai.evaluate(
    data=dataset,
    scorers=scorers,
)

Evaluating:   0%|          | 0/34 [Elapsed: 00:00, Remaining: ?] 

08:55:31 - LiteLLM:INFO: utils.py:3995 - 
LiteLLM completion() model= deepseek/deepseek-v3.2; provider = openrouter
2026-03-30 08:55:31 [INFO] LiteLLM: 
LiteLLM completion() model= deepseek/deepseek-v3.2; provider = openrouter
08:55:31 - LiteLLM:INFO: utils.py:3995 - 
LiteLLM completion() model= deepseek/deepseek-v3.2; provider = openrouter
08:55:31 - LiteLLM:INFO: utils.py:3995 - 
LiteLLM completion() model= deepseek/deepseek-v3.2; provider = openrouter
2026-03-30 08:55:31 [INFO] LiteLLM: 
LiteLLM completion() model= deepseek/deepseek-v3.2; provider = openrouter
2026-03-30 08:55:31 [INFO] LiteLLM: 
LiteLLM completion() model= deepseek/deepseek-v3.2; provider = openrouter
08:55:31 - LiteLLM:INFO: utils.py:3995 - 
LiteLLM completion() model= deepseek/deepseek-v3.2; provider = openrouter
2026-03-30 08:55:31 [INFO] LiteLLM: 
LiteLLM completion() model= deepseek/deepseek-v3.2; provider = openrouter
08:55:31 - LiteLLM:INFO: utils.py:3995 - 
LiteLLM completion() model= deepseek/deepseek-v3.2

In [8]:
from summit_sim.agents.action_responder import ActionRequest, action_response_agent


async def optimize_wrapper(**kwargs) -> dict:  # noqa: ANN003
    """Parse dict into pydantic request to pass to agent."""
    # 1. Pack MLflow's individual kwargs into your Pydantic model
    request_data = ActionRequest(**kwargs)

    # 2. Await your traced LangGraph agent
    response = await action_response_agent(input_data=request_data)

    # 3. Dump the Pydantic response to a dictionary for MLflow compatibility
    return response.model_dump()

In [9]:
import os

from mlflow.genai.optimize.optimizers import MetaPromptOptimizer

os.environ["MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION"] = "True"

result = mlflow.genai.optimize_prompts(
    predict_fn=optimize_wrapper,
    train_data=dataset,
    prompt_uris=[
        "prompts:/action-responder-system@latest",
        "prompts:/action-responder-user@latest",
    ],
    optimizer=MetaPromptOptimizer(
        reflection_model=JUDGE_MODEL_ENDPOINT,
        guidelines="This prompt is for a Wilderness First Responder agent which dynamically responds to students actions in an emergency situation",
    ),
    scorers=scorers,
)

/home/bhamm/repos/summit-sim/.venv/lib/python3.12/site-packages/mlflow/data/dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(
2026/03/30 08:45:59 INFO mlflow.genai.optimize.optimizers.metaprompt_optimizer: 34 training examples provided, using few-shot metaprompting
2026/03/30 08:45:59 INFO mlflow.genai.optimize.optimizers.metaprompt_optimizer: Evaluating baseline prompts on training data...
2026/03/30 08:46:00 INFO mlflow.models.evaluation.utils.trace: Auto tracing is temporarily enabled during the model evaluation for computing some metrics and debugging. To disable tracing, call `mlflow.autolog(disable=True)`.
2026-03-30 08:46:01 [INFO] summit_sim.agents.action_responder: Processing student action: turn=2/5, action_length=3
2026-03-30 08:46:01 [INFO] summit_s

🏃 View run learned-dolphin-580 at: https://mlflow.bhamm-lab.com/#/experiments/7/runs/4763757cfe9a40e2965324bea550fabe
🧪 View experiment at: https://mlflow.bhamm-lab.com/#/experiments/7


[Trace(trace_id=tr-1131ab4e77883c2d68781d27ba6d2e5b), Trace(trace_id=tr-be78bbe938363d1b52b5baf460ba2ab8), Trace(trace_id=tr-fdf96697b33d862b18d905226a92b8b5), Trace(trace_id=tr-f03cf5c8599f148a161b69071b58a899), Trace(trace_id=tr-7526bdd95bbf3931cc300d5a75aa179b), Trace(trace_id=tr-e17e4017d6b0ce2dbb89893b9ab24762), Trace(trace_id=tr-ed2d76a275b02081c7d2efa49d418c7e), Trace(trace_id=tr-ae202b2ee184e1cb09a8f6283e3168da), Trace(trace_id=tr-c7601a5c5614ca9d1dc5284c7fa13572), Trace(trace_id=tr-46b7b8717164ff1520299495e2f33e37)]

In [ ]:
optimized_system_prompt = result.optimized_prompts[0]
optimized_user_prompt = result.optimized_prompts[1]

print(f"Optimized system prompt URI: {optimized_system_prompt.uri}")
print(f"Optimized system template: {optimized_system_prompt.template}")
print(f"Optimized user prompt URI: {optimized_user_prompt.uri}")
print(f"Optimized user template: {optimized_user_prompt.template}")

In [ ]:
models_to_test = [
    "google/gemini-3.1-flash-lite-preview",
    "qwen/qwen3.5-flash-02-23",
    "qwen/qwen3-235b-a22b-2507",
    "openai/gpt-4o-mini",
    "openai/gpt-4.1-mini",
    "openai/gpt-5-nano",
    "openai/gpt-5.4-nano",
    "openai/gpt-oss-120b",
    "deepseek/deepseek-v3.2",
    "x-ai/grok-4.1-fast",
    "xiaomi/mimo-v2-flash",
    "z-ai/glm-4.7-flash",
    "",
]